In [1]:
%load_ext cuml.accel
%run /workspace/alvin/SAR_ML/notebooks/SSR/SSRtransforms_preloaded.py
import os
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.optim.lr_scheduler import CosineAnnealingLR, OneCycleLR
from torch.utils.data import DataLoader, ConcatDataset
import re
import copy
import torchvision
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from cuml.manifold import TSNE, UMAP
from joblib import Parallel, delayed
from tqdm import tqdm
import pickle

/opt/py_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [36]:
def set_seed(seed=42):
    """Set all random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Uncomment only if you need 100% determinism and can handle errors
    # torch.use_deterministic_algorithms(True, warn_only=True)
    # os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    
    os.environ["PYTHONHASHSEED"] = str(seed)

def worker_init_fn(worker_id):
    """DataLoader worker init for reproducibility"""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

In [3]:
workspace = "/workspace/alvin/SAR_ML"
# workspace = "/mnt/d/Users/Admin/Projects/dso/SAR_ML"
data_workspace = os.path.join(workspace, "data/SAMPLE")
# clutter_dir = os.path.join(workspace, "data/MSTAR/CLUTTER/15_DEG")

In [13]:
with open(os.path.join(workspace, "weights/SSR/SAMPLE_synth_gmm_cache.pkl"), "rb") as f:
    gmm_cache_cp = pickle.load(f)

In [14]:
# use for dso server

gmm_cache = dict()
for key, value in gmm_cache_cp.items():
    new_key = key.replace("/mnt/d/Users/Admin/Projects/dso/SAR_ML", workspace)
    gmm_cache[new_key] = value

In [16]:
synth_ds = DatasetFolderWithPath(
    os.path.join(data_workspace, "mat_files/synth"),
    extensions = (".mat"),
    transform = transforms.Compose([Magnitude(), LogMapping(c = 1000.0), NumpyToTensor3Channel()]),
    loader = mat_file_loader
)

SSR_synth_ds = DatasetFolderWithPath(
    os.path.join(data_workspace, "mat_files/synth"),
    extensions = (".mat"),
    transform = transforms.Compose([Magnitude(), LogMapping(c = 1000.0), SSRAugmentation(gmm_cache, alpha=0.6, beta=0.4, apply_prob=0.5, gaussian_noise = True, mu_s = 0.0, sigma_s = 0.3), NumpyToTensor3Channel()]),
    loader = mat_file_loader
)

meas_ds = DatasetFolderWithPath(
    os.path.join(data_workspace, "mat_files/real"),
    extensions = (".mat"),
    transform = transforms.Compose([Magnitude(), LogMapping(c = 1000.0), NumpyToTensor3Channel()]),
    loader = mat_file_loader
)

In [26]:
def _get_features_from_model(model, dataset, device):
    """
    parameters:
    model: PyTorch model to extract features from
    dataset: PyTorch Dataset to extract features for
    device: torch.device to perform computations on

    returns:
    features: numpy array of shape (num_samples, feature_dim) containing extracted features
    labels: numpy array of shape (num_samples,) containing corresponding labels
    """
    model = model.to(device)
    model.eval()

    feature_extractor = nn.Sequential(*list(model.children())[:-1])
    feature_extractor = feature_extractor.to(device)
    feature_extractor.eval()

    features, labels_list = [], []

    with torch.no_grad():
        for inputs, targets in DataLoader(
            dataset, batch_size = 32, 
            shuffle = False, num_workers = 8,
            pin_memory = True, persistent_workers=True, 
            worker_init_fn=worker_init_fn, generator=torch.Generator().manual_seed(seed)
            ):

            inputs = inputs.to(device)
            feats = feature_extractor(inputs)
            feats = feats.view(feats.size(0), -1)
            features.append(feats.cpu())
            labels_list.append(targets)

    features = torch.cat(features).numpy()
    labels = torch.cat(labels_list).numpy()

    return features, labels

In [27]:
def _gaussian_fit(features, labels, dataset):
    """
    Fit a Gaussian distribution to the features of a specific class.

    parameters:
    features: cupy array of shape (num_samples, feature_dim) containing extracted features
    labels: cupy array of shape (num_samples,) containing corresponding labels
    dataset: PyTorch Dataset containing the class information

    returns:
    class_mu: numpy array of shape (feature_dim,) containing the mean vector of the fitted Gaussian
    sigma_inv: numpy array of shape (feature_dim, feature_dim) containing the inverse covariance matrix of the fitted Gaussian
    """

    class_paras = {}
    for class_idx in dataset.class_to_idx.values():
        targets = (labels == class_idx)
        class_features = features[targets]

        class_mu = class_features.mean(axis = 0)
        class_paras[class_idx] = class_mu

    sigma = np.cov(features, rowvar = False) # rowvar means observations are rows, features are columns
    sigma_inv = np.linalg.inv(sigma)

    return class_paras, sigma_inv

In [28]:
def _mahalanobis_distance(feature, class_paras, sigma_inv):
    """
    Compute the Mahalanobis distance between a feature vector (usually test samples) and a class mean.

    parameters:
    feature: numpy array of shape (feature_dim,) containing the feature vector
    class_paras: dictionary mapping class indices to their mean vectors
    sigma_inv: numpy array of shape (feature_dim, feature_dim) containing the inverse covariance matrix

    returns:
    distance: vector representing the Mahalanobis distance from class distributions (shape of (num_classes,))
    """
    dist_vector = np.zeros(len(class_paras))
    for class_idx, class_mu in class_paras.items():
        diff = feature - class_mu
        dist_vector[class_idx] = np.sqrt(diff.T @ sigma_inv @ diff)
    return dist_vector

In [29]:
def _mahalanobis_distance_for_dataset(features, class_paras, sigma_inv):
    """
    Compute the Mahalanobis distance for a set of features against class distributions.

    parameters:
    features: numpy array of shape (num_samples, feature_dim) containing feature vectors
    class_paras: dictionary mapping class indices to their mean vectors
    sigma_inv: numpy array of shape (feature_dim, feature_dim) containing the inverse covariance matrix

    returns:
    distances: numpy array of shape (num_samples, num_classes) containing Mahalanobis distances to each class
    """
    num_samples = features.shape[0]
    num_classes = len(class_paras)
    distances = np.zeros((num_samples, num_classes))

    for i in range(num_samples):
        distances[i, :] = _mahalanobis_distance(features[i], class_paras, sigma_inv)

    return distances

In [37]:
for x in range(2):
    test_acc_lst = []
    device = torch.cuda.set_device("cuda:0")
    seed_lst = [10, 42, 100, 123, 666, 777, 849, 1000, 1111, 1234]
    for i, seed in enumerate(seed_lst):
        set_seed(seed)
        model = models.resnet18(weights = None) # dont load ImageNet Weights
        # new_model.fc = nn.Linear(new_model.fc.in_features, len(train_ds.class_to_idx))
        model.fc = nn.Sequential(
            nn.Dropout(p = 0.4),
            nn.Linear(model.fc.in_features, len(synth_ds.class_to_idx))
        )
        # Load your trained weights
        model.load_state_dict(torch.load(
            os.path.join(workspace, f"weights/SSR/Experiment_1/SSR_exp1_full_runs/rn18_seed{seed}_b16.pth"),
            map_location=device
        ))

        # this gives the synth_features and synth_labels
        SSR_synth_features, SSR_synth_labels = _get_features_from_model(model, SSR_synth_ds, device)
        SSR_synth_class_paras, SSR_synth_sigma_inv = _gaussian_fit(SSR_synth_features, SSR_synth_labels, SSR_synth_ds)

        meas_features, meas_labels = _get_features_from_model(model, meas_ds, device)
        meas_distances = _mahalanobis_distance_for_dataset(meas_features, SSR_synth_class_paras, SSR_synth_sigma_inv)

        pred_class = np.argmin(meas_distances, axis = 1)
        test_acc = (pred_class == meas_labels).sum() / len(meas_labels)
        print(f"Seed: {seed}, Test Accuracy: {test_acc:.4f}")
        test_acc_lst.append(test_acc)

Seed: 10, Test Accuracy: 0.8981
Seed: 42, Test Accuracy: 0.9219
Seed: 100, Test Accuracy: 0.8952
Seed: 123, Test Accuracy: 0.9063
Seed: 666, Test Accuracy: 0.8840
Seed: 777, Test Accuracy: 0.9026
Seed: 849, Test Accuracy: 0.9175
Seed: 1000, Test Accuracy: 0.9048
Seed: 1111, Test Accuracy: 0.8996
Seed: 1234, Test Accuracy: 0.9175
Seed: 10, Test Accuracy: 0.8981
Seed: 42, Test Accuracy: 0.9219
Seed: 100, Test Accuracy: 0.8952
Seed: 123, Test Accuracy: 0.9063
Seed: 666, Test Accuracy: 0.8840
Seed: 777, Test Accuracy: 0.9026
Seed: 849, Test Accuracy: 0.9175
Seed: 1000, Test Accuracy: 0.9048
Seed: 1111, Test Accuracy: 0.8996
Seed: 1234, Test Accuracy: 0.9175


In [38]:
test_acc_arr = np.array(test_acc_lst)
print(f"Min: {test_acc_arr.min() * 100:.4f}")
print(f"Max: {test_acc_arr.max() * 100:.4f}")
print(f"Avg, Std: {test_acc_arr.mean() * 100:.4f}, {test_acc_arr.std() * 100:.4f}")

Min: 88.4015
Max: 92.1933
Avg, Std: 90.4758, 1.1035


In [39]:
for x in range(2):
    test_acc_lst = []
    device = torch.device("cuda")
    seed_lst = [10, 42, 100, 123, 666, 777, 849, 1000, 1111, 1234]
    for i, seed in enumerate(seed_lst):
        model = models.resnet18(weights = None) # dont load ImageNet Weights
        # new_model.fc = nn.Linear(new_model.fc.in_features, len(train_ds.class_to_idx))
        model.fc = nn.Sequential(
            nn.Dropout(p = 0.4),
            nn.Linear(model.fc.in_features, len(synth_ds.class_to_idx))
        )
        # Load your trained weights
        model.load_state_dict(torch.load(
            os.path.join(workspace, f"weights/SSR/Experiment_1/wo_aug_exp1_full_runs/rn18_seed{seed}_b16.pth"),
            map_location=device
        ))

        # this gives the synth_features and synth_labels
        synth_features, synth_labels = _get_features_from_model(model, synth_ds, device)
        synth_class_paras, synth_sigma_inv = _gaussian_fit(synth_features, synth_labels, synth_ds)

        meas_features, meas_labels = _get_features_from_model(model, meas_ds, device)
        meas_distances = _mahalanobis_distance_for_dataset(meas_features, synth_class_paras, synth_sigma_inv)

        pred_class = np.argmin(meas_distances, axis = 1)
        test_acc = (pred_class == meas_labels).sum() / len(meas_labels)
        print(f"Seed: {seed}, Test Accuracy: {test_acc:.4f}")
        test_acc_lst.append(test_acc)

Seed: 10, Test Accuracy: 0.8439
Seed: 42, Test Accuracy: 0.8468
Seed: 100, Test Accuracy: 0.8431
Seed: 123, Test Accuracy: 0.8558
Seed: 666, Test Accuracy: 0.8297
Seed: 777, Test Accuracy: 0.8669
Seed: 849, Test Accuracy: 0.8506
Seed: 1000, Test Accuracy: 0.8595
Seed: 1111, Test Accuracy: 0.8476
Seed: 1234, Test Accuracy: 0.8706
Seed: 10, Test Accuracy: 0.8439
Seed: 42, Test Accuracy: 0.8468
Seed: 100, Test Accuracy: 0.8431
Seed: 123, Test Accuracy: 0.8558
Seed: 666, Test Accuracy: 0.8297
Seed: 777, Test Accuracy: 0.8669
Seed: 849, Test Accuracy: 0.8506
Seed: 1000, Test Accuracy: 0.8595
Seed: 1111, Test Accuracy: 0.8476
Seed: 1234, Test Accuracy: 0.8706


In [41]:
test_acc_arr = np.array(test_acc_lst)
print(f"Min: {test_acc_arr.min() * 100:.4f}")
print(f"Max: {test_acc_arr.max() * 100:.4f}")
print(f"Avg, Std: {test_acc_arr.mean() * 100:.4f}, {test_acc_arr.std() * 100:.4f}")

Min: 82.9740
Max: 87.0632
Avg, Std: 85.1450, 1.1512
